Project: Unicorn Industry Analysis (2019–2021)

Objective:
Identify the top three industries that produced the highest number of new
unicorn companies between 2019 and 2021. For each of these industries,
summarize the yearly number of new unicorns and their average valuation
(in billions of USD).

Business Question:
Which industries generated the most new unicorns during 2019–2021, and how
did their yearly unicorn count and average valuation compare?

Tables Used:
- dates
    company_id     : Unique company identifier
    date_joined    : Date the company achieved unicorn status
    year_founded   : Year the company was founded

- industries
    company_id     : Unique company identifier
    industry       : Industry classification

- funding
    company_id     : Unique company identifier
    valuation      : Company valuation (USD)

Methodology:
1. Identify the three industries with the highest number of companies that
   became unicorns between 2019 and 2021.
2. Retrieve each qualifying company's unicorn year.
3. Join funding information to obtain company valuations.
4. Calculate, for each industry and year:
      - Number of unicorns
      - Average valuation (billions of USD)
5. Sort results by year (descending) and unicorn count (descending).

Output Columns:
- industry                  : Industry name
- year                      : Year the company became a unicorn
- num_unicorns              : Number of unicorns created that year
- avg_valuation_billions    : Average company valuation (USD billions)

In [5]:
-- Step 1: Identify the three best-performing industries based on the number of unicorns created in 2019-2021
WITH top_3 AS (
	SELECT 
		i.industry,
		COUNT(i.*) AS num_unicorns
	FROM dates AS d
	RIGHT JOIN industries AS i
	ON i.company_id = d.company_id
	LEFT JOIN companies AS c
	ON c.company_id = i.company_id
	WHERE DATE_PART('year', d.date_joined) IN (2019, 2020, 2021)
	GROUP BY i.industry
	ORDER BY num_unicorns DESC
	LIMIT 3
	),

-- Step 2: Join top_3 CTE with the industry and dates tables to only keep the top 3 best-performing industries while acquiring the year joined for each company_id
unicorn_year AS (
	SELECT
		i.industry,
		i.company_id,
		DATE_PART('year', d.date_joined) AS year
	FROM industries AS i
	INNER JOIN top_3 AS t
	ON i.industry = t.industry
	INNER JOIN dates AS d
	ON d.company_id = i.company_id
	WHERE DATE_PART('year', d.date_joined) IN (2019, 2020, 2021)
	),

-- Step 3: Join the unicorn_year CTE with the funding table to get the valuation for each company_id
valuations AS (
	SELECT 
		u.*, 
		f.valuation/1000000000 AS valuation_billions
	FROM unicorn_year AS u
	LEFT JOIN funding AS f
	ON u.company_id = f.company_id
	)

-- Step 4: Calculate the average valuation and the number of unicorns for the top 3 industries in each year (2019, 2020, 2021)
SELECT 
	industry, 
	year, 
	COUNT(company_id) AS num_unicorns,
	ROUND(AVG(valuation_billions), 2) AS average_valuation_billions
FROM valuations
GROUP BY industry, year
ORDER BY year DESC, num_unicorns DESC

,industry,year,num_unicorns,average_valuation_billions
0,Fintech,2021,138,2.75
1,Internet software & services,2021,119,2.15
2,E-commerce & direct-to-consumer,2021,47,2.47
3,Internet software & services,2020,20,4.35
4,E-commerce & direct-to-consumer,2020,16,4.00
5,Fintech,2020,15,4.33
6,Fintech,2019,20,6.80
7,Internet software & services,2019,13,4.23
8,E-commerce & direct-to-consumer,2019,12,2.58
